In [1]:
# Check that the libraries we need are installed

import pandas    # pandas: reads and handles tables of data 
import sklearn   # scikit-learn: machine learning models, train/test split, error metrics
import joblib    # joblib: saves a trained model to a file so the app can reuse it

print("ok", sklearn.__version__)


ok 1.9.0


In [2]:
# Load the listings
# Real Flatfox listings downloaded by Paul (see Flatfox_protocollAPI.ipynb)
import pandas as pd

df = pd.read_csv("data/flatfox_zurich.csv")

print(df.shape)   # (number of rows, number of columns)
df.head()         # first 5 rows

(1411, 9)


,id,municipality,rooms,surface_m2,year_built,lat,lon,rent_gross,url
0,86544292,Oetwil an der Limmat,4.0,65.0,NaN,47.424243,8.402642,2880.0,https://flatfox.ch/en/flat/schweizackerstrasse...
1,86541961,Bergdietikon,5.5,130.0,2013.0,47.386955,8.392162,3360.0,https://flatfox.ch/en/flat/im-fohret-9-8962-be...
2,86543395,Rottenschwil,4.5,160.0,NaN,47.315915,8.361502,3550.0,https://flatfox.ch/en/flat/8919-rottenschwil/8...
3,86543178,Zufikon,5.5,160.0,2007.0,47.345664,8.358112,3990.0,https://flatfox.ch/en/flat/5621-zufikon/86543178/
4,86542785,Dietikon,4.5,92.0,1973.0,47.401805,8.398952,3450.0,https://flatfox.ch/en/flat/8953-dietikon/86542...


In [3]:
# Basic statistics for each numeric column (count, mean, min, max, ...)
df.describe()

,id,rooms,surface_m2,year_built,lat,lon,rent_gross
count,1.411000e+03,1411.000000,1411.000000,654.000000,1411.000000,1411.000000,1411.000000
mean,8.590515e+07,3.809709,88.754075,2018.813456,47.388460,8.540692,3638.683912
std,6.020749e+06,1.098567,44.078473,706.797715,0.043329,0.080322,1360.381378
min,1.389523e+06,1.000000,15.000000,1448.000000,47.280130,8.353572,690.000000
25%,8.633328e+07,3.000000,60.000000,1969.000000,47.363719,8.496732,2690.000000
50%,8.645602e+07,4.000000,80.000000,2010.000000,47.387634,8.537522,3450.000000
75%,8.651291e+07,4.500000,101.000000,2024.000000,47.414230,8.578607,4260.000000
max,8.654429e+07,6.500000,300.000000,20026.000000,47.499384,8.747352,9990.000000


In [4]:
# Data quality check
print("Missing values per column:")
print(df.isna().sum())
print("Duplicate listing ids:", df["id"].duplicated().sum())

Missing values per column:
id                0
municipality      0
rooms             0
surface_m2        0
year_built      757
lat               0
lon               0
rent_gross        0
url               0
dtype: int64
Duplicate listing ids: 0


In [5]:
# Average rent per municipality
groups = df.groupby("municipality")       # 1. split the table into one group per municipality
rents = groups["rent_gross"]              # 2. keep only the rent column
average_rent = rents.mean()               # 3. average rent of each group
average_rent.sort_values(ascending=False) # 4. most expensive first

municipality
Kilchberg                9470.0
Erlenbach                6930.0
Uetliberg                6560.0
Weiningen                5810.0
zürich                   5630.0
                          ...  
Adlikon b. Regensdorf    2240.0
Dällikon                 2235.0
Stallikon                2230.0
Birmensdorf              1720.0
Pfaffhausen              1150.0
Name: rent_gross, Length: 121, dtype: float64

In [6]:
#  Average rent per m² per municipality 

df["m2_price"] = df["rent_gross"] / df["surface_m2"]       # 1. new column: CHF per m² for each flat
groups = df.groupby("municipality")                        # 2. regroup so the new column is included                               
avg_rent_m2 = groups["m2_price"].mean()                    # 3. average CHF/m² per municipality
avg_rent_m2.sort_values(ascending=False).round(2)          # 4. most expensive first (rounded)

municipality
Pfaffhausen            76.67
Zürich-Seebach         70.86
Regensdorf / Watt      66.81
Glattpark (Opfikon)    62.99
Wipkingen, Zurich      59.55
                       ...  
Ottenbach              22.61
Rottenschwil           22.19
Greifensee             21.17
Illnau                 19.89
Jonen                  18.71
Name: m2_price, Length: 121, dtype: float64

In [7]:
# Outlier check: look at the cheapest and most expensive listings per m2
cols = ["municipality", "rooms", "surface_m2", "rent_gross", "m2_price"]
print(df["m2_price"].describe().round(1))
print(df.sort_values("m2_price")[cols].head(5))   # cheapest per m2
print(df.sort_values("m2_price")[cols].tail(5))   # most expensive per m2

# Keep only plausible prices. Above 80 CHF/m2 we find tiny flats (23-26 m2)
# for 3'400-4'700 CHF: probably furnished or business apartments.
before = len(df)
df = df[df["m2_price"].between(10, 80)].copy()
print("Removed:", before - len(df), "| Remaining:", len(df))

count    1411.0
mean       46.3
std        18.4
min        12.9
25%        33.8
50%        43.5
75%        54.3
max       181.2
Name: m2_price, dtype: float64
     municipality  rooms  surface_m2  rent_gross   m2_price
624    Regensdorf    4.0       140.0      1800.0  12.857143
1101       Zürich    3.0       140.0      2020.0  14.428571
1086       Zürich    4.5       200.0      3310.0  16.550000
52          Jonen    4.5       143.0      2390.0  16.713287
1384    Gutenswil    5.5       160.0      2750.0  17.187500
     municipality  rooms  surface_m2  rent_gross    m2_price
493        Zürich    2.0        23.0      3380.0  146.956522
511        Zürich    3.5        25.0      3780.0  151.200000
794        Zürich    2.0        23.0      3560.0  154.782609
1081       Zürich    2.0        24.0      4030.0  167.916667
1078       Zürich    2.0        26.0      4710.0  181.153846
Removed: 64 | Remaining: 1347


In [8]:
# year_built is missing for about half of the listings.
# We mark which ones are missing, then fill them later with the training median.
df["year_missing"] = df["year_built"].isna().astype(int)

# Explicit list of input columns: the app must use the same ones, in the same order
FEATURES = ["rooms", "surface_m2", "year_built", "year_missing", "lat", "lon"]

X = df[FEATURES]
y = df["m2_price"]
print(X.columns)

Index(['rooms', 'surface_m2', 'year_built', 'year_missing', 'lat', 'lon'], dtype='str')


In [9]:
# Random Forest: predicts the price per m2, then rent = price per m2 x m2
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Split the data: 70% training, 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
X_train = X_train.copy()
X_test = X_test.copy()

# Fill missing years with the median of the TRAINING data only,
# so no information from the test apartments leaks into the model
YEAR_MEDIAN = X_train["year_built"].median()
X_train["year_built"] = X_train["year_built"].fillna(YEAR_MEDIAN)
X_test["year_built"] = X_test["year_built"].fillna(YEAR_MEDIAN)
print("Median year used for missing values:", YEAR_MEDIAN)

# Create and train the model (random_state makes the results reproducible)
model = RandomForestRegressor(random_state=1)
model.fit(X_train, y_train)

# Predict the price per m2 of the test apartments (the model has never seen them)
y_pred = model.predict(X_test)

# Convert price per m2 into monthly rent in CHF
surface = X_test["surface_m2"]
real_rent = y_test * surface
predicted_rent = y_pred * surface

# How far are the estimated rents from the real ones, in CHF?
mae = mean_absolute_error(real_rent, predicted_rent)        # average error
rmse = root_mean_squared_error(real_rent, predicted_rent)   # big errors count more

print("MAE: ", round(mae), "CHF")
print("RMSE:", round(rmse), "CHF")

Median year used for missing values: 2012.0
MAE:  530 CHF
RMSE: 736 CHF


In [10]:
# Baseline: a simple method without machine learning.
# Estimated rent = average price per m2 of the municipality x m2 of the apartment.
# If our Random Forest is not better than this, the model is not worth using.

# 1. Split the full table with the same parameters as the model.
#    X has no municipality and no rent, but the baseline needs both, so we split df.
#    Same test_size and random_state -> same 405 test apartments as the model,
#    so the two errors can be compared fairly.
train_df, test_df = train_test_split(df, test_size=0.3, random_state=1)

# 2. Average price per m2 of each municipality.
#    Training apartments only: the test rents must stay unseen, like for the model.
avg_price_m2 = train_df.groupby("municipality")["m2_price"].mean()

# 3. Estimated rent of each test apartment: municipality average per m2 x m2.
# Some test apartments are in a municipality with no training listings.
# For those we use the average price per m2 of all training apartments.
overall_avg = train_df["m2_price"].mean()
baseline_price = test_df["municipality"].map(avg_price_m2).fillna(overall_avg)
print("Test flats without their own municipality average:",
      test_df["municipality"].map(avg_price_m2).isna().sum())

baseline_rent = baseline_price * test_df["surface_m2"]

# 4. Mean Absolute Error (MAE): for each test apartment, the difference between
#    real and estimated rent (without sign), then the average.
#    Same metric as the model, so the two numbers can be compared directly.
mae_baseline = mean_absolute_error(test_df["rent_gross"], baseline_rent)

print("Baseline MAE:", round(mae_baseline), "CHF")
print("Model MAE:   ", round(mae), "CHF")

Test flats without their own municipality average: 22
Baseline MAE: 902 CHF
Model MAE:    530 CHF


In [11]:
# Save the trained model so the app can use it
import joblib

joblib.dump(model, "model.joblib")

['model.joblib']

### Result: model vs. baseline

| Method | MAE (average error) |
|---|---|
| Baseline: average price per m² of the municipality × m² | 902 CHF |
| Random Forest | **531 CHF** |

We used 1,347 real Flatfox listings (64 listings with an unrealistic price per m² were removed). On the test apartments, the Random Forest is about 40% more precise than the baseline, because it also uses rooms, year of construction and exact location.

Missing years of construction were filled with the median year of the training data (2012).

The model's MAE (about 530 CHF) is used as the tolerance for the "fair" verdict in the app.